In [ ]:
# Load Jupyter-compatible Dash extension for inline UI rendering
from jupyter_dash import JupyterDash

# Import layout elements, tables, charts, and map visualization packages
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Load operating system routines
import os

# Load math and data handling libraries
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Pull in the custom CRUD module built for database interaction
# Ensure CRUD_Python_Module.py is in the local directory
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# Authenticate using the restricted aacuser credentials configured in Module Three
# Defaults to localhost:27017, database 'aac', collection 'animals'
username = "aacuser"
password = "password123"

# Open the database connection through the CRUD module
db = AnimalShelter(username, password)

# Rescue profiles transcribed from the Dashboard Specifications Document
# (Rescue Type and Preferred Dog Breeds table); keys match the radio item values
RESCUE_PROFILES = {
    "water": {
        "label": "Water Rescue",
        "breeds": ["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"],
        "sex": "Intact Female",
        "min_weeks": 26, "max_weeks": 156,
    },
    "mountain": {
        "label": "Mountain or Wilderness Rescue",
        "breeds": ["German Shepherd", "Alaskan Malamute", "Old English Sheepdog",
                   "Siberian Husky", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 26, "max_weeks": 156,
    },
    "disaster": {
        "label": "Disaster or Individual Tracking",
        "breeds": ["Doberman Pinscher", "German Shepherd", "Golden Retriever",
                   "Bloodhound", "Rottweiler"],
        "sex": "Intact Male",
        "min_weeks": 20, "max_weeks": 300,
    },
}

# Breed matching mode for the rescue queries
# "exact"    - breed must equal a listed name (e.g. "German Shepherd")
# "contains" - breed only needs to contain a listed name (also returns "German Shepherd Mix")
BREED_MATCH_MODE = "contains"


def build_query(profile, mode=BREED_MATCH_MODE):
    """Assemble the MongoDB query document for one rescue profile."""
    if mode == "contains":
        # Escaped regex patterns match the breed name anywhere in the field
        breed_filter = {"$in": [re.compile(re.escape(b)) for b in profile["breeds"]]}
    else:
        breed_filter = {"$in": profile["breeds"]}

    return {
        "animal_type": "Dog",
        "breed": breed_filter,
        "sex_upon_outcome": profile["sex"],
        "age_upon_outcome_in_weeks": {"$gte": profile["min_weeks"], "$lte": profile["max_weeks"]},
    }


def fetch_records(query):
    """Run a read query and return a dataframe with the ObjectId column removed."""
    records = pd.DataFrame.from_records(db.read(query))
    # Drop the MongoDB internal ObjectId column to prevent data table rendering crashes;
    # errors='ignore' keeps an empty result from failing on the missing column
    return records.drop(columns=['_id'], errors='ignore')


# Execute a read query with an empty filter to pull all records into the starting dataset
df = fetch_records({})

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Encode the Grazioso Salvare logo for inline display in the page header
image_filename = 'Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

app.layout = html.Div([
    html.Div(id='hidden-div', style={'display': 'none'}),

    # Header block: client logo linked to the client home page, title, and creator credit
    html.Div(style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center', 'gap': '30px'},
             children=[
        html.A(
            html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()),
                     style={'height': '130px'}),
            href='https://www.snhu.edu', target='_blank'
        ),
        html.Div([
            html.H1('SNHU CS-340 Dashboard', style={'margin': '0'}),
            # Unique identifier crediting the dashboard creator
            html.H3('Grazioso Salvare Dashboard - Jacobsin Enriquez', style={'margin': '5px 0 0 0'})
        ])
    ]),
    html.Hr(),

    # Interactive filter options; each choice runs a rescue-type query against MongoDB
    html.Div([
        html.B('Filter by rescue type: '),
        dcc.RadioItems(
            id='filter-type',
            options=[{'label': p['label'], 'value': key} for key, p in RESCUE_PROFILES.items()]
                    + [{'label': 'Reset', 'value': 'reset'}],
            value='reset',  # Start in the unfiltered state
            labelStyle={'display': 'inline-block', 'marginRight': '20px'}
        )
    ]),
    html.Hr(),

    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        # Table interaction controls configured for client navigation
        editable=False,             # Read-only grid to protect source integrity
        filter_action="native",     # Built-in search/filter per column
        filter_query='',            # No column filter at start; cleared on each rescue filter
        sort_action="native",       # Enable column header sorting
        sort_mode="multi",          # Support multi-column sorting queues
        sort_by=[],                 # No sort at start; cleared on each rescue filter
        column_selectable="single", # Single column highlight mode
        selected_columns=[],        # Clear initial column focus
        row_selectable="single",    # Lock selection to one row for map sync
        row_deletable=False,        # Prevent accidental row deletion in view
        selected_rows=[0],          # Default to first row so map populates immediately
        page_action="native",       # Enable pagination controls
        page_current=0,             # Index to first page
        page_size=10,               # Limit view to 10 entries per frame
        style_table={'overflowX': 'auto'},  # Horizontal scroll for wide records
        style_cell={'textAlign': 'left', 'minWidth': '100px', 'padding': '5px'},
        style_header={'fontWeight': 'bold'}
    ),
    html.Br(),
    html.Hr(),

    # Chart row: breed chart and geolocation chart side by side
    # Bottom padding so the charts do not clip against the page edge
    html.Div(className='row',
             style={'display': 'flex', 'paddingBottom': '50px'},
             children=[
        html.Div(id='graph-id', className='col s12 m6', style={'width': '45%'}),
        html.Div(id='map-id', className='col s12 m6', style={'width': '55%'})
    ])
])

#############################################
# Interaction Between Components / Controller
#############################################
# Apply the selected rescue filter: re-query MongoDB and reset the table view
# Also resets row selection, page, column filters, and sorting so the map and chart
# start from the first record of the new result set
@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'selected_rows'),
     Output('datatable-id', 'page_current'),
     Output('datatable-id', 'filter_query'),
     Output('datatable-id', 'sort_by')],
    [Input('filter-type', 'value')],
    prevent_initial_call=True  # Starting data is already loaded in the layout
)
def update_dashboard(filter_type):
    # Rescue types map to their profile query; Reset falls through to the unfiltered query
    if filter_type in RESCUE_PROFILES:
        query = build_query(RESCUE_PROFILES[filter_type])
    else:
        query = {}

    dff = fetch_records(query)

    # No matches: clear the table and selection instead of pointing at a missing row
    if dff.empty:
        return [], [], 0, '', []

    return dff.to_dict('records'), [0], 0, '', []


# Breed breakdown chart driven by the rows currently shown in the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    # Skip rendering if data stream has not initialized or the filter returned nothing
    if not viewData:
        return [html.P("No records match the current filter.")]

    dff = pd.DataFrame.from_dict(viewData)

    # Count animals per breed and keep the ten most common for readability
    breed_counts = dff['breed'].value_counts().head(10).reset_index()
    breed_counts.columns = ['breed', 'count']  # Normalize column names across pandas versions

    fig = px.bar(breed_counts, x='count', y='breed', orientation='h',
                 title=f'Top Breeds in Current View ({len(dff)} animals)')
    # Single series in the Grazioso Salvare brand red; counts labeled at the bar ends
    fig.update_traces(marker_color='#B5123A', texttemplate='%{x}', textposition='outside',
                      hovertemplate='%{y}: %{x} animals<extra></extra>')
    fig.update_layout(yaxis={'categoryorder': 'total ascending', 'title': ''},
                      xaxis_title='Number of animals', plot_bgcolor='white',
                      height=500, margin=dict(l=10, r=40, t=60, b=40))
    fig.update_xaxes(showgrid=True, gridcolor='#EEEEEE')

    return [dcc.Graph(figure=fig)]


# Highlight selected columns in the data table
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Bypass styling loop if no columns are active
    if not selected_columns:
        return []
    return [{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# Bind table row selection to map marker updates
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # Skip rendering if data stream has not initialized or the filter returned nothing
    if not viewData:
        return []

    dff = pd.DataFrame.from_dict(viewData)

    # Resolve active row index; default to index 0 if unselected or filtered out
    if not index:
        row = 0
    else:
        row = index[0]

    # Pull the selected record and read fields by column name, not position,
    # so the map keeps working if the column order ever changes
    record = dff.iloc[row]
    lat, lon = record['location_lat'], record['location_long']
    name = record['name'] if record['name'] else "No name on record"

    # Show a message instead of a map if the record is missing coordinates
    if pd.isna(lat) or pd.isna(lon):
        return [html.P(f"No location on record for {name}.")]

    # Render Leaflet map centered on the selected animal; a unique map id per animal
    # forces a fresh map so the view re-centers on every selection
    return [
        dl.Map(id=f"map-{record['animal_id']}",
               style={'width': '100%', 'height': '500px'},
               center=[lat, lon], zoom=11, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with hover tooltip (name and breed) and a click popup
            # holding the candidate details staff need for screening
            dl.Marker(position=[lat, lon],
                      children=[
                dl.Tooltip(f"{name} - {record['breed']}"),
                dl.Popup([
                    html.H3(name),
                    html.P([html.B("Breed: "), record['breed']]),
                    html.P([html.B("Age: "), record['age_upon_outcome']]),
                    html.P([html.B("Sex: "), record['sex_upon_outcome']]),
                    html.P([html.B("Animal ID: "), record['animal_id']])
                ])
            ])
        ])
    ]


# Launch the Dash server on default port 8050; open the proxy link printed below the cell
app.run_server()